# Normal LJ Assembly

I'll just try to implement the attractive crystallization of many particles in 3D with LJ potentials, with spatial partitioning, and see how good of a simulation I can get. This should teach me how spatial partitioning is supposed to work.

In [1]:
import jax.numpy as jnp
from jax import config
import jax
config.update("jax_enable_x64", True) # necessary for 64-bit precision

from jax import jit, random, grad, value_and_grad, remat, jacfwd, vmap, lax
from jax.example_libraries import optimizers
from jax_md import space, smap, energy, minimize, quantity, simulate, partition, rigid_body, util
# from jax_md.colab_tools import renderer
from jax_md import partition

import numpy as np

## Setup

In [118]:
dimension = 3

RADIUS = 0.5
number_of_particles = 200
volume_density = 0.01

# get box size
get_box_size = lambda phi, N, rad: (N * jnp.pi * 4 * rad**3 / phi / 3.0) ** (1/3)
box_size = float(get_box_size(volume_density, number_of_particles, RADIUS))
print("Box Size: {:.3f}".format(box_size))

# spatial functions
disp_fn, shift_fn = space.periodic(box_size)

# simulation parameters
dt = 1e-3
num_steps = int(1e5)
collect_every = 1000

max_time = num_steps * dt
print("Max Time: {:.3f} ({:.5f} step size/{:.0f} steps)".format(max_time, dt, num_steps))

kT = lambda t: jnp.where(t < max_time / 2, 1.0, 0.1)

key = random.PRNGKey(0)
key, split = random.split(key)
R = box_size * random.uniform(split, (number_of_particles, dimension), dtype=np.float64)

Box Size: 21.878
Max Time: 100.000 (0.00100 step size/100000 steps)


In [119]:
# first repel the particles using soft sphere interactions
energy_fn = energy.soft_sphere_pair(disp_fn, sigma=RADIUS*2.0)
init_fn, apply_fn = minimize.fire_descent(energy_fn, shift_fn)

state = init_fn(R)
apply_fn_jit = jit(lambda i, state: apply_fn(state))
state = lax.fori_loop(0, 5000, apply_fn_jit, state)

R = state.position

In [120]:
energy_fn = energy.lennard_jones_pair(disp_fn, sigma=RADIUS*2.0, epsilon=10.0, r_cutoff = RADIUS*2.5*2.0)
init_fn, apply_fn = simulate.nvt_langevin(energy_fn, shift_fn, kT=kT(0), dt=dt, gamma=5.0)
# apply_fn_jit = jit(lambda i, state: apply_fn(state))

state = init_fn(key, R)
trajectory = np.zeros((num_steps, number_of_particles, dimension))

def step_fn(i, state_and_trajectory):
    state, trajectory = state_and_trajectory
    trajectory = trajectory.at[i].set(state.position)
    # state = lax.fori_loop(0, collect_every, apply_fn_jit, state)

    state = apply_fn(state, kT=kT(i*dt))
    return state, trajectory

state, trajectory = lax.fori_loop(0, num_steps, step_fn, (state, trajectory))
jax.block_until_ready(trajectory)

None

In [121]:
trajectory = trajectory[::1000]

In [122]:
with open("trajectory.xyz", "w") as f:
    for t in range(trajectory.shape[0]):
        f.write(f"{number_of_particles}\n")
        f.write("Properties=species:S:1:pos:R:3:radius:R:1\n")

        for i in range(number_of_particles):
            x, y, z = trajectory[t, i]
            r = 0.5
            f.write(f"A {x} {y} {z} {r}\n")